# Creating a TTD sample

In [21]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display
from matplotlib.ticker import PercentFormatter
from sklearn.metrics import brier_score_loss, roc_auc_score

project_root = next(
    (
        folder for folder in [Path.cwd(), *Path.cwd().parents]
        if (
            folder / "src" / "origination_scorecard" / "generator.py"
        ).is_file()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not locate the project generator.")

sys.path.insert(0, str(project_root / "src"))

from origination_scorecard.generator import (
    GeneratorConfig,
    _make_applications,
    _make_performance,
)

training_path = (
    project_root / "data" / "KIGB Modelling" / "KIGB in sample.csv"
)
if not training_path.is_file():
    raise FileNotFoundError(training_path)

---

## generating the sample

In [22]:
sample_size = 2_000
random_seed = 20_261_002

config = GeneratorConfig(
    n_accounts=sample_size,
    random_seed=random_seed,
)
rng = np.random.default_rng(random_seed)

# Same application and performance-generation functions as Notebook 1.
applications = _make_applications(config, rng)
_, generated_truth = _make_performance(applications, config, rng)

# Give the independent cohort a retrospective application period.
date_rng = np.random.default_rng(random_seed + 1)
applications["application_date"] = (
    pd.Timestamp("2025-01-01")
    + pd.to_timedelta(
        date_rng.integers(0, 181, sample_size),
        unit="D",
    )
).strftime("%Y-%m-%d")

applications["account_id"] = [
    f"RETRO-2025-{i:06d}"
    for i in range(1, sample_size + 1)
]

# Keep application inputs separate from outcomes.
application_columns = [
    "account_id",
    "application_date",
    "requested_loan_amount",
    "disposable_income",
    "debt_to_income_ratio",
    "recent_application_count",
    "revolving_utilisation",
    "historic_defaults",
    "time_at_address_months",
    "loan_purpose",
]

retrospective_applicants = applications[
    application_columns
].copy()

# Retain the simulated outcome separately until after scoring.
retrospective_outcomes = applications[
    ["master_account_key", "account_id"]
].merge(
    generated_truth[
        ["master_account_key", "total_missed_payments"]
    ],
    on="master_account_key",
    validate="one_to_one",
)

missed = retrospective_outcomes["total_missed_payments"]
retrospective_outcomes["observed_bad"] = missed.ge(3).astype(int)
retrospective_outcomes["observed_group"] = np.select(
    [missed.eq(0), missed.between(1, 2), missed.ge(3)],
    ["Known good", "Indeterminate", "Known bad"],
    default="Invalid",
)

if retrospective_applicants["account_id"].duplicated().any():
    raise ValueError("Duplicate retrospective applicant IDs.")
if retrospective_outcomes["observed_group"].eq("Invalid").any():
    raise ValueError("Invalid generated payment outcome.")

print(f"Retrospective applicants: {len(retrospective_applicants):,}")
display(
    retrospective_outcomes["observed_group"]
    .value_counts()
    .rename_axis("Observed group")
    .reset_index(name="Accounts")
)

Retrospective applicants: 2,000


,Observed group,Accounts
0,Known good,1880
1,Known bad,80
2,Indeterminate,40


In [23]:
from pathlib import Path

# project_root is defined in notebook 16's first code cell.
output_folder = project_root / "data" / "TTD Modelling"
output_folder.mkdir(parents=True, exist_ok=True)

output_file = output_folder / "TTD retrospective scored.csv"

required_columns = {
    "account_id",
    "predicted_pd",
    "score",
    "observed_bad",
}

missing = required_columns - set(evaluation.columns)
if missing:
    raise ValueError(f"Complete TTD sample is missing: {sorted(missing)}")

if evaluation["account_id"].isna().any():
    raise ValueError("Some TTD accounts have no account ID.")
if evaluation["account_id"].duplicated().any():
    raise ValueError("Duplicate TTD account IDs found.")
if evaluation[["predicted_pd", "observed_bad"]].isna().any().any():
    raise ValueError("Some TTD accounts lack a PD or observed outcome.")

evaluation.to_csv(output_file, index=False)

print(f"Saved {len(evaluation):,} TTD accounts to:")
print(output_file)

Saved 2,000 TTD accounts to:
c:\Users\georg\OneDrive\Coding\Origination-Scorecard\data\TTD Modelling\TTD retrospective scored.csv


In [24]:
model_inputs = [
    "disposable_income",
    "debt_to_income_ratio",
    "recent_application_count",
    "revolving_utilisation",
    "historic_defaults",
    "time_at_address_months",
    "loan_purpose",
]

if retrospective_applicants["account_id"].duplicated().any():
    raise ValueError("Duplicate applicant IDs.")

if retrospective_applicants[model_inputs].isna().any().any():
    raise ValueError("Missing KIGB model input.")

if retrospective_applicants["time_at_address_months"].lt(0).any():
    raise ValueError("Negative address tenure.")

outcome_columns = {
    "target_bad",
    "kigb_bad",
    "outcome",
    "missed_payment_months",
    "predicted_pd",
    "score",
    "observed_bad",
    "observed_group",
}

if outcome_columns & set(retrospective_applicants.columns):
    raise ValueError(
        "Outcome or existing model result found in applicant inputs."
    )

print("The retrospective TTD applicant inputs are ready for KIGB scoring.")
print(
    "Application dates:",
    retrospective_applicants["application_date"].min(),
    "to",
    retrospective_applicants["application_date"].max(),
)

The retrospective TTD applicant inputs are ready for KIGB scoring.
Application dates: 2025-01-01 to 2025-06-30


---

## Data Cleaning

In [25]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from IPython.display import display

project_root = next(
    (
        folder
        for folder in [Path.cwd(), *Path.cwd().parents]
        if (folder / "src" / "origination_scorecard" / "generator.py").is_file()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        f"Could not locate the project from {Path.cwd()}"
    )

ttd_folder = project_root / "data" / "TTD Modelling"
source_file = ttd_folder / "TTD retrospective scored.csv"
clean_file = ttd_folder / "TTD retrospective scored clean.csv"
rejected_file = ttd_folder / "TTD retrospective rejected rows.csv"

if not source_file.is_file():
    raise FileNotFoundError(
        f"TTD dataset not found: {source_file}\n"
        "Run the save cell in notebook 16 first."
    )

ttd_raw = pd.read_csv(source_file, dtype={"account_id": "string"})

print(f"Loaded {len(ttd_raw):,} rows from:")
print(source_file)
display(ttd_raw.head())

Loaded 2,000 rows from:
c:\Users\georg\OneDrive\Coding\Origination-Scorecard\data\TTD Modelling\TTD retrospective scored.csv


,account_id,application_date,requested_loan_amount,disposable_income,debt_to_income_ratio,recent_application_count,revolving_utilisation,historic_defaults,time_at_address_months,loan_purpose,predicted_pd,score,decision,total_missed_payments,observed_group,observed_bad
0,RETRO-2025-000001,2025-02-20,7000.0,274.21,0.1490,1,0.4782,0,39,car,0.035239,582.621179,Flag / decline,0,Known good,0
1,RETRO-2025-000002,2025-04-24,8900.0,1730.16,0.0857,1,0.0756,0,136,other,0.019964,599.469448,Accept,0,Known good,0
2,RETRO-2025-000003,2025-06-08,6900.0,2333.96,0.0407,0,0.6599,0,102,car,0.028337,589.116751,Flag / decline,6,Known bad,1
3,RETRO-2025-000004,2025-04-10,16100.0,863.66,0.1702,2,0.4448,1,69,debt_consolidation,0.061068,565.973599,Flag / decline,0,Known good,0
4,RETRO-2025-000005,2025-03-01,6400.0,295.79,0.2073,1,0.1288,0,115,car,0.023389,594.800892,Accept,0,Known good,0


In [26]:
required_columns = {
    "account_id",
    "application_date",
    "loan_purpose",
    "predicted_pd",
    "score",
    "observed_bad",
}

missing_columns = required_columns - set(ttd_raw.columns)
if missing_columns:
    raise KeyError(
        f"Missing columns: {sorted(missing_columns)}"
    )

ttd = ttd_raw.copy()
ttd["account_id_original"] = ttd["account_id"]

text_columns = ttd.select_dtypes(
    include=["object", "string"]
).columns

for column in text_columns:
    ttd[column] = (
        ttd[column]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )
    ttd[column] = ttd[column].mask(ttd[column].eq(""))

# Keep loan-purpose labels consistent with the model's
# "education" indicator.
ttd["loan_purpose"] = ttd["loan_purpose"].str.lower()

print("Loan-purpose values after cleaning:")
display(ttd["loan_purpose"].value_counts(dropna=False))

Loan-purpose values after cleaning:


loan_purpose
debt_consolidation    649
car                   518
home_improvement      348
major_purchase        249
other                 134
education             102
Name: count, dtype: Int64

In [27]:
def canonical_account_id(value):
    if pd.isna(value):
        return pd.NA

    compact = re.sub(
        r"[^A-Z0-9]",
        "",
        str(value).upper(),
    )

    match = re.fullmatch(
        r"RETRO(2025)(\d{1,6})",
        compact,
    )

    if match is None:
        return pd.NA

    year, sequence = match.groups()
    return f"RETRO-{year}-{int(sequence):06d}"


ttd["account_id"] = (
    ttd["account_id"]
    .map(canonical_account_id)
    .astype("string")
)

id_audit = pd.DataFrame({
    "original": ttd["account_id_original"],
    "cleaned": ttd["account_id"],
})

print(
    "IDs that could not be matched:",
    f"{ttd['account_id'].isna().sum():,}",
)
display(
    id_audit.loc[
        id_audit["original"].ne(id_audit["cleaned"]).fillna(True)
    ].head(20)
)

IDs that could not be matched: 0


,original,cleaned


In [28]:
ttd["application_date"] = pd.to_datetime(
    ttd["application_date"],
    errors="coerce",
)

numeric_columns = [
    "requested_loan_amount",
    "disposable_income",
    "debt_to_income_ratio",
    "recent_application_count",
    "revolving_utilisation",
    "historic_defaults",
    "time_at_address_months",
    "total_missed_payments",
    "predicted_pd",
    "score",
    "observed_bad",
]

for column in numeric_columns:
    if column in ttd.columns:
        ttd[column] = pd.to_numeric(
            ttd[column],
            errors="coerce",
        )

print("Numeric columns converted:")
print([column for column in numeric_columns if column in ttd.columns])

Numeric columns converted:
['requested_loan_amount', 'disposable_income', 'debt_to_income_ratio', 'recent_application_count', 'revolving_utilisation', 'historic_defaults', 'time_at_address_months', 'total_missed_payments', 'predicted_pd', 'score', 'observed_bad']


In [29]:
issues = pd.Series(
    "",
    index=ttd.index,
    dtype="string",
)

def flag_rows(mask, reason):
    global issues
    mask = mask.fillna(False)
    issues.loc[mask] = (
        issues.loc[mask]
        + np.where(issues.loc[mask].eq(""), "", "; ")
        + reason
    )


flag_rows(
    ttd["account_id"].isna(),
    "Missing or unrecognisable account ID",
)

flag_rows(
    ttd["account_id"].notna()
    & ttd["account_id"].duplicated(keep=False),
    "Duplicate canonical account ID",
)

flag_rows(
    ttd["application_date"].isna(),
    "Invalid application date",
)

flag_rows(
    ttd["loan_purpose"].isna(),
    "Missing loan purpose",
)

flag_rows(
    ttd["predicted_pd"].isna()
    | ~ttd["predicted_pd"].between(0, 1),
    "Missing or invalid predicted PD",
)

flag_rows(
    ttd["score"].isna()
    | ~np.isfinite(ttd["score"]),
    "Missing or invalid score",
)

flag_rows(
    ttd["observed_bad"].isna()
    | ~ttd["observed_bad"].isin([0, 1]),
    "Missing or invalid observed outcome",
)

# Check the generated outcome against notebook 16's definition.
if "total_missed_payments" in ttd.columns:
    flag_rows(
        ttd["total_missed_payments"].isna()
        | ttd["total_missed_payments"].lt(0),
        "Missing or invalid missed-payment total",
    )

    comparable = (
        ttd["total_missed_payments"].notna()
        & ttd["observed_bad"].isin([0, 1])
    )
    flag_rows(
        comparable
        & ttd["observed_bad"].ne(
            ttd["total_missed_payments"].ge(3).astype(int)
        ),
        "Observed bad conflicts with missed-payment total",
    )

ttd["cleaning_issue"] = issues.mask(issues.eq(""))

print(f"Rows with at least one issue: {ttd['cleaning_issue'].notna().sum():,}")
display(
    ttd["cleaning_issue"]
    .fillna("Retained")
    .value_counts()
    .rename_axis("Result")
    .reset_index(name="Rows")
)

Rows with at least one issue: 0


,Result,Rows
0,Retained,2000


In [30]:
rejected_ttd = ttd.loc[
    ttd["cleaning_issue"].notna()
].copy()

clean_ttd = (
    ttd.loc[ttd["cleaning_issue"].isna()]
    .drop(
        columns=["account_id_original", "cleaning_issue"]
    )
    .reset_index(drop=True)
)

if clean_ttd.empty:
    raise ValueError(
        "No rows passed cleaning. Inspect rejected_ttd before saving."
    )

clean_ttd["observed_bad"] = clean_ttd["observed_bad"].astype(int)

assert clean_ttd["account_id"].is_unique
assert clean_ttd["predicted_pd"].between(0, 1).all()
assert clean_ttd["observed_bad"].isin([0, 1]).all()

clean_ttd.to_csv(clean_file, index=False)
rejected_ttd.to_csv(rejected_file, index=False)

print(f"Original rows: {len(ttd_raw):,}")
print(f"Clean rows: {len(clean_ttd):,}")
print(f"Rejected rows: {len(rejected_ttd):,}")
print(f"Clean file: {clean_file}")
print(f"Rejected-row audit: {rejected_file}")

Original rows: 2,000
Clean rows: 2,000
Rejected rows: 0
Clean file: c:\Users\georg\OneDrive\Coding\Origination-Scorecard\data\TTD Modelling\TTD retrospective scored clean.csv
Rejected-row audit: c:\Users\georg\OneDrive\Coding\Origination-Scorecard\data\TTD Modelling\TTD retrospective rejected rows.csv


---

## Refitting the regression on the TTD sample

In [31]:
kigb_train = pd.read_csv(training_path)

scaled_predictors = [
    "debt_to_income_ratio",
    "recent_application_count",
    "revolving_utilisation",
    "historic_defaults",
    "log_address_tenure",
]

numeric_inputs = [
    "debt_to_income_ratio",
    "recent_application_count",
    "revolving_utilisation",
    "historic_defaults",
    "disposable_income",
    "time_at_address_months",
]

def prepare_inputs(data):
    rows = data.copy()

    for column in numeric_inputs:
        rows[column] = pd.to_numeric(
            rows[column], errors="raise"
        )

    if rows[numeric_inputs + ["loan_purpose"]].isna().any().any():
        raise ValueError("Missing KIGB model input.")
    if not np.isfinite(rows[numeric_inputs].to_numpy()).all():
        raise ValueError("Infinite KIGB model input.")
    if rows["time_at_address_months"].lt(0).any():
        raise ValueError("Negative address tenure.")

    rows["low_disposable_income_flag"] = (
        rows["disposable_income"] < 150
    ).astype(float)
    rows["log_address_tenure"] = np.log1p(
        rows["time_at_address_months"]
    )
    return rows

train_inputs = prepare_inputs(kigb_train)
retrospective_inputs = prepare_inputs(
    retrospective_applicants
)

# Scaling parameters come from the development sample only.
means = train_inputs[scaled_predictors].mean()
stds = train_inputs[scaled_predictors].std(ddof=0)

if not np.isfinite(stds).all() or (stds == 0).any():
    raise ValueError("Invalid predictor variation.")

def design_matrix(rows):
    x = (rows[scaled_predictors] - means) / stds
    x.insert(0, "const", 1.0)
    x["low_disposable_income_flag"] = rows[
        "low_disposable_income_flag"
    ]
    x["loan_purpose=education"] = (
        rows["loan_purpose"] == "education"
    ).astype(float)
    return x.astype(float)

x_train = design_matrix(train_inputs)
x_retrospective = design_matrix(
    retrospective_inputs
)[x_train.columns]

y_train = train_inputs["kigb_bad"].astype(int)

kigb_model = sm.Logit(y_train, x_train).fit(
    maxiter=200, disp=False
)
if not kigb_model.mle_retvals.get("converged", False):
    raise RuntimeError("KIGB model did not converge.")

print("Model fitted on KIGB development data only.")

Model fitted on KIGB development data only.


---

## Predict the PD and log scores (TTD) & create the benchmark score

In [32]:
evaluation = retrospective_applicants.copy()
evaluation["predicted_pd"] = np.asarray(
    kigb_model.predict(x_retrospective)
)

# Your existing score scale.
factor = 20 / np.log(2)
offset = 600 - factor * np.log(50)

def pd_to_score(probability):
    p = np.clip(
        np.asarray(probability, dtype=float),
        1e-12,
        1 - 1e-12,
    )
    return offset + factor * np.log((1 - p) / p)

evaluation["score"] = pd_to_score(
    evaluation["predicted_pd"]
)

benchmark_pd = 0.025
cutoff_score = float(pd_to_score([benchmark_pd])[0])

evaluation["decision"] = np.where(
    evaluation["score"] >= cutoff_score,
    "Accept",
    "Flag / decline",
)

# Only now attach the independently generated outcomes.
evaluation = evaluation.merge(
    retrospective_outcomes[
        [
            "account_id",
            "total_missed_payments",
            "observed_group",
            "observed_bad",
        ]
    ],
    on="account_id",
    how="left",
    validate="one_to_one",
)

if evaluation["observed_bad"].isna().any():
    raise ValueError("A retrospective outcome did not link.")

print(f"2.5% PD benchmark → minimum score {cutoff_score:.1f}")
display(
    evaluation[
        [
            "account_id", "predicted_pd", "score",
            "decision", "observed_group",
        ]
    ].head(10).style.format({
        "predicted_pd": "{:.2%}",
        "score": "{:.1f}",
    })
)

2.5% PD benchmark → minimum score 592.8


,account_id,predicted_pd,score,decision,observed_group
0,RETRO-2025-000001,3.52%,582.6,Flag / decline,Known good
1,RETRO-2025-000002,2.00%,599.5,Accept,Known good
2,RETRO-2025-000003,2.83%,589.1,Flag / decline,Known bad
3,RETRO-2025-000004,6.11%,566.0,Flag / decline,Known good
4,RETRO-2025-000005,2.34%,594.8,Accept,Known good
5,RETRO-2025-000006,2.51%,592.7,Flag / decline,Known good
6,RETRO-2025-000007,3.00%,587.4,Flag / decline,Known good
7,RETRO-2025-000008,9.30%,552.8,Flag / decline,Known good
8,RETRO-2025-000009,7.88%,558.1,Flag / decline,Indeterminate
9,RETRO-2025-000010,12.18%,544.1,Flag / decline,Known good


---

## Evaluate the actual generated results around the cutoff

In [33]:
def performance_summary(group):
    expected_bads = group["predicted_pd"].sum()
    observed_bads = int(group["observed_bad"].sum())

    return pd.Series({
        "Accounts": len(group),
        "Expected bads": expected_bads,
        "Observed bads": observed_bads,
        "Predicted bad rate": group["predicted_pd"].mean(),
        "Observed bad rate": group["observed_bad"].mean(),
        "Calibration gap": (
            group["observed_bad"].mean()
            - group["predicted_pd"].mean()
        ),
        "Observed / expected": (
            observed_bads / expected_bads
            if expected_bads > 0 else np.nan
        ),
        "Brier score": brier_score_loss(
            group["observed_bad"],
            group["predicted_pd"],
        ),
    })

accepted = evaluation.loc[
    evaluation["decision"].eq("Accept")
]
flagged = evaluation.loc[
    evaluation["decision"].eq("Flag / decline")
]

summary = pd.DataFrame({
    "All applicants": performance_summary(evaluation),
    "Accepted": performance_summary(accepted),
    "Flagged / declined": performance_summary(flagged),
}).T

display(
    summary.style.format({
        "Accounts": "{:,.0f}",
        "Expected bads": "{:.1f}",
        "Observed bads": "{:,.0f}",
        "Predicted bad rate": "{:.2%}",
        "Observed bad rate": "{:.2%}",
        "Calibration gap": "{:+.2%}",
        "Observed / expected": "{:.2f}",
        "Brier score": "{:.4f}",
    })
)

print(f"Approval rate: {len(accepted) / len(evaluation):.1%}")
print(
    "ROC AUC on this retrospective synthetic sample: "
    f"{roc_auc_score(evaluation['observed_bad'], evaluation['predicted_pd']):.3f}"
)

,Accounts,Expected bads,Observed bads,Predicted bad rate,Observed bad rate,Calibration gap,Observed / expected,Brier score
All applicants,"2,000",77.1,80,3.85%,4.00%,+0.15%,1.04,0.0383
Accepted,364,7.4,10,2.03%,2.75%,+0.71%,1.35,0.0268
Flagged / declined,"1,636",69.7,70,4.26%,4.28%,+0.02%,1.00,0.0408


Approval rate: 18.2%
ROC AUC on this retrospective synthetic sample: 0.589


---

# Comparing this to different cutoffs

In [34]:
from sklearn.metrics import brier_score_loss

pd_cutoffs = {
    "2.00% PD cutoff": 0.0200,
    "2.25% PD cutoff": 0.0225,
}

def calibration_summary(data):
    """Compare predicted PDs with observed 3+ missed-payment outcomes."""
    expected_bads = data["predicted_pd"].sum()
    observed_bads = int(data["observed_bad"].sum())

    return {
        "Accounts": len(data),
        "Expected bads": expected_bads,
        "Observed bads": observed_bads,
        "Predicted bad rate": data["predicted_pd"].mean(),
        "Observed bad rate": data["observed_bad"].mean(),
        "Calibration gap": (
            data["observed_bad"].mean()
            - data["predicted_pd"].mean()
        ),
        "Observed / expected": (
            observed_bads / expected_bads
            if expected_bads > 0 else np.nan
        ),
        "Brier score": brier_score_loss(
            data["observed_bad"],
            data["predicted_pd"],
        ),
    }

calibration_reports = {}

for cutoff_name, cutoff_pd in pd_cutoffs.items():
    # Accept accounts below the individual PD cutoff.
    accepted = evaluation.loc[
        evaluation["predicted_pd"] < cutoff_pd
    ].copy()

    flagged = evaluation.loc[
        evaluation["predicted_pd"] >= cutoff_pd
    ].copy()

    report = pd.DataFrame({
        "All applicants": calibration_summary(evaluation),
        "Accepted": calibration_summary(accepted),
        "Flagged / declined": calibration_summary(flagged),
    }).T

    calibration_reports[cutoff_name] = report

    print(f"\n{cutoff_name}")
    print(
        f"Approval rate: {len(accepted) / len(evaluation):.1%} "
        f"({len(accepted):,} of {len(evaluation):,})"
    )

    display(
        report.style.format({
            "Accounts": "{:,.0f}",
            "Expected bads": "{:.1f}",
            "Observed bads": "{:,.0f}",
            "Predicted bad rate": "{:.2%}",
            "Observed bad rate": "{:.2%}",
            "Calibration gap": "{:+.2%}",
            "Observed / expected": "{:.2f}",
            "Brier score": "{:.4f}",
        })
    )


2.00% PD cutoff
Approval rate: 5.1% (103 of 2,000)


,Accounts,Expected bads,Observed bads,Predicted bad rate,Observed bad rate,Calibration gap,Observed / expected,Brier score
All applicants,"2,000",77.1,80,3.85%,4.00%,+0.15%,1.04,0.0383
Accepted,103,1.4,3,1.37%,2.91%,+1.55%,2.13,0.0285
Flagged / declined,"1,897",75.7,77,3.99%,4.06%,+0.07%,1.02,0.0388



2.25% PD cutoff
Approval rate: 10.4% (208 of 2,000)


,Accounts,Expected bads,Observed bads,Predicted bad rate,Observed bad rate,Calibration gap,Observed / expected,Brier score
All applicants,"2,000",77.1,80,3.85%,4.00%,+0.15%,1.04,0.0383
Accepted,208,3.7,7,1.77%,3.37%,+1.60%,1.90,0.0327
Flagged / declined,"1,792",73.4,73,4.10%,4.07%,-0.02%,0.99,0.0389


In [35]:
comparison = []

for cutoff_name, cutoff_pd in pd_cutoffs.items():
    accepted = evaluation.loc[
        evaluation["predicted_pd"] < cutoff_pd
    ]

    comparison.append({
        "PD cutoff": cutoff_name,
        "Accepted accounts": len(accepted),
        "Approval rate": len(accepted) / len(evaluation),
        "Expected bads": accepted["predicted_pd"].sum(),
        "Observed bads": int(accepted["observed_bad"].sum()),
        "Predicted accepted bad rate": accepted["predicted_pd"].mean(),
        "Observed accepted bad rate": accepted["observed_bad"].mean(),
        "Calibration gap": (
            accepted["observed_bad"].mean()
            - accepted["predicted_pd"].mean()
        ),
    })

display(
    pd.DataFrame(comparison).style.format({
        "Approval rate": "{:.1%}",
        "Expected bads": "{:.1f}",
        "Predicted accepted bad rate": "{:.2%}",
        "Observed accepted bad rate": "{:.2%}",
        "Calibration gap": "{:+.2%}",
    })
)

,PD cutoff,Accepted accounts,Approval rate,Expected bads,Observed bads,Predicted accepted bad rate,Observed accepted bad rate,Calibration gap
0,2.00% PD cutoff,103,5.1%,1.4,3,1.37%,2.91%,+1.55%
1,2.25% PD cutoff,208,10.4%,3.7,7,1.77%,3.37%,+1.60%


---

## Saving the dataset

In [36]:
available_dataframes = {
    name: dataframe
    for name, dataframe in list(globals().items())
    if isinstance(dataframe, pd.DataFrame)
}

scored_ttd = None
source_dataframe_name = None

possible_outcomes = [
    "kigb_bad",
    "target_bad",
    "observed_bad",
]

# Preferred DataFrame names first
preferred_names = [
    "evaluation",
    "through_the_door_scored",
    "scored_applicants",
    "ttd_scored",
    "policy_sample",
]

for name in preferred_names:
    if name in available_dataframes:
        dataframe = available_dataframes[name]

        has_pd = "predicted_pd" in dataframe.columns
        has_outcome = any(
            column in dataframe.columns
            for column in possible_outcomes
        )

        if has_pd and has_outcome:
            scored_ttd = dataframe.copy()
            source_dataframe_name = name
            break

# Search all other DataFrames if a preferred one was not found
if scored_ttd is None:
    for name, dataframe in available_dataframes.items():
        has_pd = "predicted_pd" in dataframe.columns
        has_outcome = any(
            column in dataframe.columns
            for column in possible_outcomes
        )

        if has_pd and has_outcome:
            scored_ttd = dataframe.copy()
            source_dataframe_name = name
            break

if scored_ttd is None:
    raise NameError(
        "No DataFrame contains both 'predicted_pd' and a true outcome. "
        "Run the TTD scoring and outcome-merging cells before saving."
    )

# Standardise the outcome name
if "kigb_bad" not in scored_ttd.columns:
    if "target_bad" in scored_ttd.columns:
        scored_ttd["kigb_bad"] = scored_ttd["target_bad"]
    elif "observed_bad" in scored_ttd.columns:
        scored_ttd["kigb_bad"] = scored_ttd["observed_bad"]

print(f"Dataset found: {source_dataframe_name}")
print(f"Accounts: {len(scored_ttd):,}")
print(f"Columns: {len(scored_ttd.columns):,}")

Dataset found: evaluation
Accounts: 2,000
Columns: 17


In [37]:
required_columns = [
    "predicted_pd",
    "kigb_bad",
]

missing_columns = [
    column
    for column in required_columns
    if column not in scored_ttd.columns
]

if missing_columns:
    raise KeyError(
        f"Cannot save the scored TTD dataset. "
        f"Missing columns: {missing_columns}"
    )

scored_ttd["predicted_pd"] = pd.to_numeric(
    scored_ttd["predicted_pd"],
    errors="coerce",
)

scored_ttd["kigb_bad"] = pd.to_numeric(
    scored_ttd["kigb_bad"],
    errors="coerce",
)

if scored_ttd[required_columns].isna().any().any():
    raise ValueError(
        "Missing or invalid predicted PD/outcome values were found."
    )

if not scored_ttd["predicted_pd"].between(0, 1).all():
    raise ValueError("predicted_pd must be between 0 and 1.")

if not scored_ttd["kigb_bad"].isin([0, 1]).all():
    raise ValueError("kigb_bad must contain only 0 and 1.")

scored_ttd["kigb_bad"] = scored_ttd["kigb_bad"].astype(int)

if (
    "account_id" in scored_ttd.columns
    and scored_ttd["account_id"].duplicated().any()
):
    raise ValueError("Duplicate account IDs were found.")

# Save the completed dataset
scored_ttd.to_csv(
    ttd_file,
    index=False,
)

print("TTD dataset saved successfully.")
print("Location:", ttd_file)
print("Rows saved:", f"{len(scored_ttd):,}")

NameError: name 'ttd_file' is not defined